**Install Feast**

In [1]:
!pip -q install feast==0.64.0 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10,

In [2]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Upload the Skill Gap Dataset**

In [3]:
from google.colab import files

uploaded = files.upload()

Saving CSE_Skill_Gap_Dataset_1000.csv to CSE_Skill_Gap_Dataset_1000.csv


In [4]:
import pandas as pd

df = pd.read_csv("CSE_Skill_Gap_Dataset_1000.csv")

print(df.head())
print(df.shape)


    StudentName  CurriculumCoverage  IndustryDemand  JobFrequency  \
0   Kiran Patel                  74              81             2   
1    Diya Singh                  83              75            10   
2  Ananya Verma                  46              59             1   
3  Sneha Sharma                  60              91             5   
4   Arjun Naidu                  59              75            10   

   ExpertImportance  Programming  DataStructures  Algorithms  Database  \
0                 2           58              63          46        80   
1                 5           63              64          49        87   
2                 4           57              63          98        46   
3                 1           72              94          97        83   
4                 2           43             100          69        45   

   WebDevelopment  ...  Communication  ProblemSolving  Teamwork  Projects  \
0              41  ...             79              58        48

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Examine the Data**

In [5]:
print(df.columns)
print(df.info())
print(df.isnull().sum())


Index(['StudentName', 'CurriculumCoverage', 'IndustryDemand', 'JobFrequency',
       'ExpertImportance', 'Programming', 'DataStructures', 'Algorithms',
       'Database', 'WebDevelopment', 'Cloud', 'AI_ML', 'Communication',
       'ProblemSolving', 'Teamwork', 'Projects', 'Internship',
       'Certifications', 'CurriculumAlignment', 'IndustryRequirement',
       'SkillGap', 'GapStatus'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   StudentName          1000 non-null   object
 1   CurriculumCoverage   1000 non-null   int64 
 2   IndustryDemand       1000 non-null   int64 
 3   JobFrequency         1000 non-null   int64 
 4   ExpertImportance     1000 non-null   int64 
 5   Programming          1000 non-null   int64 
 6   DataStructures       1000 non-null   int64 
 7   Algorithms           1000 non-null 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Create Features- perform simple feature engineering.**

In [7]:
import pandas as pd

# Entity
df["student_id"] = range(1, len(df) + 1)

# Handle missing values
numeric_columns = [
    "Programming", "DataStructures", "Algorithms",
    "Database", "WebDevelopment", "Cloud", "AI_ML",
    "Communication", "ProblemSolving", "Teamwork", "Projects",
    "Internship", "Certifications", "CurriculumAlignment",
    "IndustryRequirement", "SkillGap"
]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Rename / cast features
# 'CGPA' column does not exist, removing this line
df["programming"] = df["Programming"].astype("int64")
df["data_structures"] = df["DataStructures"].astype("int64")
df["algorithms"] = df["Algorithms"].astype("int64")
df["database"] = df["Database"].astype("int64")
df["web_development"] = df["WebDevelopment"].astype("int64")
df["cloud"] = df["Cloud"].astype("int64")
df["ai_ml"] = df["AI_ML"].astype("int64")
df["communication"] = df["Communication"].astype("int64")
df["problem_solving"] = df["ProblemSolving"].astype("int64")
df["teamwork"] = df["Teamwork"].astype("int64")
df["projects"] = df["Projects"].astype("int64")
df["internship"] = df["Internship"].astype("int64")
df["certifications"] = df["Certifications"].astype("int64")
df["curriculum_alignment"] = df["CurriculumAlignment"].astype("int64")
df["industry_requirement"] = df["IndustryRequirement"].astype("int64")
df["skill_gap"] = df["SkillGap"].astype("int64")

# Derived feature 1: average technical skill
df["technical_skill_avg"] = (
    df["programming"] + df["data_structures"] + df["algorithms"]
    + df["database"] + df["web_development"] + df["cloud"] + df["ai_ml"]
) / 7
df["technical_skill_avg"] = df["technical_skill_avg"].astype("float32")

# Derived feature 2: average professional skill
df["professional_skill_avg"] = (
    df["communication"] + df["problem_solving"] + df["teamwork"]
) / 3
df["professional_skill_avg"] = df["professional_skill_avg"].astype("float32")

# Target
df["placement_status"] = df["GapStatus"].astype("int64")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
df[
    [
        "student_id", "programming", "data_structures",
        "algorithms", "database", "web_development", "cloud", "ai_ml",
        "communication", "problem_solving", "teamwork", "projects",
        "internship", "certifications", "curriculum_alignment",
        "industry_requirement", "skill_gap",
        "technical_skill_avg", "professional_skill_avg",
        "placement_status"
    ]
].head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,programming,data_structures,algorithms,database,web_development,cloud,ai_ml,communication,problem_solving,teamwork,projects,internship,certifications,curriculum_alignment,industry_requirement,skill_gap,technical_skill_avg,professional_skill_avg,placement_status
0,1,58,63,46,80,41,62,58,79,58,48,3,0,3,76,84,8,58.285713,61.666668,0
1,2,63,64,49,87,54,94,42,80,44,45,3,0,0,78,78,0,64.714287,56.333332,0
2,3,57,63,98,46,69,39,95,94,76,40,1,1,2,44,56,12,66.714287,70.000000,1
3,4,72,94,97,83,75,40,88,95,72,42,3,1,1,57,86,29,78.428574,69.666664,1
4,5,43,100,69,45,71,51,88,72,42,54,2,0,1,57,79,22,66.714287,56.000000,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Add Event Timestamps**

Feast models feature values with timestamps so that historical retrieval can perform point-in-time joins. The Skill Gap dataset is not naturally a time-series dataset, so artificial timestamps are created for this classroom demonstration.


**Separate Features and Labels**

In [21]:
import pandas as pd

# Add event timestamps (moved from cell E7uo_70r0fIb to ensure columns exist)
base_time = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time + pd.to_timedelta(df["student_id"], unit="s")
)

df["created_timestamp"] = (
    df["event_timestamp"] + pd.Timedelta(seconds=1)
)

feature_df = df[
    [
        "student_id", "event_timestamp", "created_timestamp",
        "programming", "data_structures", "algorithms",
        "database", "web_development", "cloud", "ai_ml",
        "communication", "problem_solving", "teamwork", "projects",
        "internship", "certifications", "curriculum_alignment",
        "industry_requirement", "skill_gap",
        "technical_skill_avg", "professional_skill_avg"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
label_df = df[
    ["student_id", "event_timestamp", "placement_status"]
].copy()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:
print("Feature data:")
display(feature_df.head())

print("Labels:")
display(label_df.head())

Feature data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,created_timestamp,programming,data_structures,algorithms,database,web_development,cloud,ai_ml,...,problem_solving,teamwork,projects,internship,certifications,curriculum_alignment,industry_requirement,skill_gap,technical_skill_avg,professional_skill_avg
0,1,2026-01-01 00:00:01+00:00,2026-01-01 00:00:02+00:00,58,63,46,80,41,62,58,...,58,48,3,0,3,76,84,8,58.285713,61.666668
1,2,2026-01-01 00:00:02+00:00,2026-01-01 00:00:03+00:00,63,64,49,87,54,94,42,...,44,45,3,0,0,78,78,0,64.714287,56.333332
2,3,2026-01-01 00:00:03+00:00,2026-01-01 00:00:04+00:00,57,63,98,46,69,39,95,...,76,40,1,1,2,44,56,12,66.714287,70.000000
3,4,2026-01-01 00:00:04+00:00,2026-01-01 00:00:05+00:00,72,94,97,83,75,40,88,...,72,42,3,1,1,57,86,29,78.428574,69.666664
4,5,2026-01-01 00:00:05+00:00,2026-01-01 00:00:06+00:00,43,100,69,45,71,51,88,...,42,54,2,0,1,57,79,22,66.714287,56.000000


Labels:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,placement_status
0,1,2026-01-01 00:00:01+00:00,0
1,2,2026-01-01 00:00:02+00:00,0
2,3,2026-01-01 00:00:03+00:00,1
3,4,2026-01-01 00:00:04+00:00,1
4,5,2026-01-01 00:00:05+00:00,1


**Create the Feast Repository**

In [16]:
import os

repo_path = "/content/skill_gap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [17]:
import os

repo_path = "/content/skill_gap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [18]:
feature_df.to_parquet(
    f"{repo_path}/data/skill_gap_features.parquet",
    index=False
)


Create feature_store.yaml

In [19]:
feature_store_yaml = """
project: skill_gap_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)


**Define the Entity and Feature View** Create:features.py

In [22]:
feature_definition = """
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE student"
)

skill_gap_source = FileSource(
    name="skill_gap_source",
    path="data/skill_gap_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

skill_gap_feature_view = FeatureView(
    name="skill_gap_features",
    entities=[student],
    ttl=timedelta(days=50000),
    schema=[
        Field(name="programming", dtype=Int64),
        Field(name="data_structures", dtype=Int64),
        Field(name="algorithms", dtype=Int64),
        Field(name="database", dtype=Int64),
        Field(name="web_development", dtype=Int64),
        Field(name="cloud", dtype=Int64),
        Field(name="ai_ml", dtype=Int64),
        Field(name="communication", dtype=Int64),
        Field(name="problem_solving", dtype=Int64),
        Field(name="teamwork", dtype=Int64),
        Field(name="projects", dtype=Int64),
        Field(name="internship", dtype=Int64),
        Field(name="certifications", dtype=Int64),
        Field(name="curriculum_alignment", dtype=Int64),
        Field(name="industry_requirement", dtype=Int64),
        Field(name="skill_gap", dtype=Int64),
        Field(name="technical_skill_avg", dtype=Float32),
        Field(name="professional_skill_avg", dtype=Float32),
    ],
    source=skill_gap_source,
    online=True
)

skill_gap_feature_service = FeatureService(
    name="skill_gap_placement_service",
    features=[skill_gap_feature_view]
)
"""

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

In [23]:
!find /content/skill_gap_feast -maxdepth 2 -type f


/content/skill_gap_feast/data/skill_gap_features.parquet
/content/skill_gap_feast/features.py
/content/skill_gap_feast/feature_store.yaml


**Apply the Feature Definitions**

In [24]:
%cd /content/skill_gap_feast


/content/skill_gap_feast


In [25]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [26]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [27]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

**Create the Feast Store Object**

In [28]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/skill_gap_feast"
)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
feature_service = store.get_feature_service(
    "skill_gap_placement_service"
)


**Retrieve Historical Features**

In [30]:
entity_df = label_df.copy()

display(entity_df.head())

,student_id,event_timestamp,placement_status
0,1,2026-01-01 00:00:01+00:00,0
1,2,2026-01-01 00:00:02+00:00,0
2,3,2026-01-01 00:00:03+00:00,1
3,4,2026-01-01 00:00:04+00:00,1
4,5,2026-01-01 00:00:05+00:00,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [31]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
display(training_data.head())

,student_id,event_timestamp,placement_status,programming,data_structures,algorithms,database,web_development,cloud,ai_ml,...,problem_solving,teamwork,projects,internship,certifications,curriculum_alignment,industry_requirement,skill_gap,technical_skill_avg,professional_skill_avg
0,1,2026-01-01 00:00:01+00:00,0,58,63,46,80,41,62,58,...,58,48,3,0,3,76,84,8,58.285713,61.666668
1,2,2026-01-01 00:00:02+00:00,0,63,64,49,87,54,94,42,...,44,45,3,0,0,78,78,0,64.714287,56.333332
2,3,2026-01-01 00:00:03+00:00,1,57,63,98,46,69,39,95,...,76,40,1,1,2,44,56,12,66.714287,70.000000
3,4,2026-01-01 00:00:04+00:00,1,72,94,97,83,75,40,88,...,72,42,3,1,1,57,86,29,78.428574,69.666664
4,5,2026-01-01 00:00:05+00:00,1,43,100,69,45,71,51,88,...,42,54,2,0,1,57,79,22,66.714287,56.000000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Train a Model Using Feast Features**

In [34]:
feature_columns = [
    "programming", "data_structures", "algorithms",
    "database", "web_development", "cloud", "ai_ml",
    "communication", "problem_solving", "teamwork", "projects",
    "internship", "certifications", "curriculum_alignment",
    "industry_requirement", "skill_gap",
    "technical_skill_avg", "professional_skill_avg"
]

In [35]:
X = training_data[feature_columns]
y = training_data["placement_status"]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [37]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [38]:
predictions = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 100.0 %


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The important point is not the accuracy. The model received its feature matrix through Feast historical feature retrieval.

**Materialize Features to the Online Store**

In [40]:
%cd /content/skill_gap_feast

!feast materialize \
    2026-01-01T00:00:00 \
    2026-01-02T00:00:00


/content/skill_gap_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib

**Retrieve Online Features**

In [42]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": 25}
    ]
).to_dict()


In [44]:
print(online_features)

{'student_id': [25], 'programming': [63], 'problem_solving': [83], 'skill_gap': [49], 'algorithms': [94], 'technical_skill_avg': [71.0], 'projects': [2], 'cloud': [55], 'ai_ml': [95], 'web_development': [79], 'teamwork': [66], 'certifications': [2], 'professional_skill_avg': [68.0], 'database': [51], 'internship': [0], 'data_structures': [60], 'curriculum_alignment': [47], 'industry_requirement': [96], 'communication': [55]}


In [43]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)

,student_id,programming,problem_solving,skill_gap,algorithms,technical_skill_avg,projects,cloud,ai_ml,web_development,teamwork,certifications,professional_skill_avg,database,internship,data_structures,curriculum_alignment,industry_requirement,communication
0,25,63,83,49,94,71.0,2,55,95,79,66,2,68.0,51,0,60,47,96,55


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Make an Online Prediction**

In [45]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [46]:
X_online = online_df[
    feature_columns
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [47]:
prediction = final_model.predict(
    X_online
)

print(
    "Predicted placement status:",
    prediction[0]
)


Predicted placement status: 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Retrieve Features for Multiple Students**

In [48]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": 10},
        {"student_id": 20},
        {"student_id": 30},
        {"student_id": 40}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)


,student_id,programming,problem_solving,skill_gap,algorithms,technical_skill_avg,projects,cloud,ai_ml,web_development,teamwork,certifications,professional_skill_avg,database,internship,data_structures,curriculum_alignment,industry_requirement,communication
0,10,52,51,-22,96,53.428570,5,43,41,44,73,0,57.666668,52,0,46,83,61,49
1,20,52,40,-5,69,59.285713,4,40,96,44,73,0,61.666668,59,1,55,73,68,72
2,30,55,92,-24,59,55.142857,4,87,24,44,82,0,91.333336,41,1,76,96,72,100
3,40,77,88,33,52,59.714287,1,33,50,79,61,1,75.000000,82,0,45,42,75,76


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [49]:
online_df["predicted_placement_status"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

display(
    online_df[
        [
            "student_id",
            "predicted_placement_status"
        ]
    ]
)


,student_id,predicted_placement_status
0,10,0
1,20,0
2,30,0
3,40,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
